In [1]:
import numpy as np
print(np.__version__)

1.23.5


In [2]:
import tensorflow as tf
print(tf.__version__)

2.10.0


In [ ]:
import mediapipe as mp
print(mp.__version__)

In [4]:
pip uninstall protobuf -y

Found existing installation: protobuf 3.19.6
Uninstalling protobuf-3.19.6:
  Successfully uninstalled protobuf-3.19.6
Note: you may need to restart the kernel to use updated packages.


You can safely remove it manually.


In [5]:
pip install protobuf==3.20.3

  Using cached protobuf-3.20.3-cp310-cp310-win_amd64.whl.metadata (698 bytes)
Using cached protobuf-3.20.3-cp310-cp310-win_amd64.whl (904 kB)
Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorboard 2.10.1 requires protobuf<3.20,>=3.9.2, but you have protobuf 3.20.3 which is incompatible.
tensorflow 2.10.0 requires protobuf<3.20,>=3.9.2, but you have protobuf 3.20.3 which is incompatible.


In [ ]:
pip uninstall mediapipe 


In [ ]:
!pip install mediapipe

In [ ]:
import cv2
import numpy as np
import os
from matplotlib import pyplot as plt
import time
import mediapipe as mp
import tensorflow as tf

In [ ]:
. Keypoints using MP Holistic

In [ ]:
mp_holistic = mp.solutions.holistic
mp_drawing = mp.solutions.drawing_utils

In [ ]:
print("OpenCV:", cv2.__version__)
print("NumPy:", np.__version__)
print("TensorFlow:", tf.__version__)
print("MediaPipe:", mp.__version__)

In [ ]:
def mediapipe_detection(image, model):
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB) # BGR 2 RGB
    image.flags.writeable = False                  # Image not writeable
    results = model.process(image)                 # Prediction
    image.flags.writeable = True                   # Image writeable
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR) # RGB 2 BGR
    return image, results

In [ ]:
def draw_landmarks(image, results):
    mp_drawing.draw_landmarks(
        image,
        results.face_landmarks,
        mp_holistic.FACE_CONNECTIONS
    )

    mp_drawing.draw_landmarks(
        image,
        results.pose_landmarks,
        mp_holistic.POSE_CONNECTIONS
    )

    mp_drawing.draw_landmarks(
        image,
        results.left_hand_landmarks,
        mp_holistic.HAND_CONNECTIONS
    )

    mp_drawing.draw_landmarks(
        image,
        results.right_hand_landmarks,
        mp_holistic.HAND_CONNECTIONS
    )

In [ ]:
def draw_styled_landmarks(image, results):
    # Draw face connections
    mp_drawing.draw_landmarks(
        image,
        results.face_landmarks,
        mp.solutions.face_mesh.FACEMESH_TESSELATION,
        mp_drawing.DrawingSpec(color=(80,110,10), thickness=1, circle_radius=1),
        mp_drawing.DrawingSpec(color=(80,256,121), thickness=1, circle_radius=1)
    )

    # Draw pose connections
    mp_drawing.draw_landmarks(
        image,
        results.pose_landmarks,
        mp_holistic.POSE_CONNECTIONS,
        mp_drawing.DrawingSpec(color=(80,22,10), thickness=2, circle_radius=4),
        mp_drawing.DrawingSpec(color=(80,44,121), thickness=2, circle_radius=2)
    )

    # Draw left hand connections
    mp_drawing.draw_landmarks(
        image,
        results.left_hand_landmarks,
        mp_holistic.HAND_CONNECTIONS,
        mp_drawing.DrawingSpec(color=(121,22,76), thickness=2, circle_radius=4),
        mp_drawing.DrawingSpec(color=(121,44,250), thickness=2, circle_radius=2)
    )

    # Draw right hand connections
    mp_drawing.draw_landmarks(
        image,
        results.right_hand_landmarks,
        mp_holistic.HAND_CONNECTIONS,
        mp_drawing.DrawingSpec(color=(245,117,66), thickness=2, circle_radius=4),
        mp_drawing.DrawingSpec(color=(245,66,230), thickness=2, circle_radius=2)
    )

In [ ]:
cap = cv2.VideoCapture(0)

# Set mediapipe model
with mp_holistic.Holistic(
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
) as holistic:

    while cap.isOpened():

        # Read feed
        ret, frame = cap.read()

        if not ret:
           break

        # Make detections
        image, results = mediapipe_detection(frame, holistic)
        print(results)

        # Draw landmarks
        draw_styled_landmarks(image, results)

        # Show to screen
        cv2.imshow('OpenCV Feed', image)

        # Break gracefully
        if cv2.waitKey(10) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

In [ ]:
# Make detections
image, results = mediapipe_detection(frame, holistic)

# Draw landmarks
draw_styled_landmarks(image, results)

# Show screen
cv2.imshow('OpenCV Feed', image)

In [ ]:
def extract_keypoints(results):

    pose = np.array([[res.x, res.y, res.z, res.visibility] 
                     for res in results.pose_landmarks.landmark]).flatten() \
            if results.pose_landmarks else np.zeros(33*4)

    face = np.array([[res.x, res.y, res.z] 
                     for res in results.face_landmarks.landmark]).flatten() \
            if results.face_landmarks else np.zeros(468*3)

    lh = np.array([[res.x, res.y, res.z] 
                   for res in results.left_hand_landmarks.landmark]).flatten() \
            if results.left_hand_landmarks else np.zeros(21*3)

    rh = np.array([[res.x, res.y, res.z] 
                   for res in results.right_hand_landmarks.landmark]).flatten() \
            if results.right_hand_landmarks else np.zeros(21*3)

    return np.concatenate([pose, face, lh, rh])

In [ ]:
result_test = extract_keypoints(results)
print(result_test)

In [ ]:
np.save('test_keypoints.npy', result_test)

In [ ]:
loaded = np.load('test_keypoints.npy')
print(loaded)

In [ ]:
import os
import numpy as np

# Base folder to store dataset
DATA_PATH = os.path.join('MP_Data')

# Actions (sign language classes)
actions = np.array(['hello', 'thanks', 'iloveyou'])

# Number of videos per action
no_sequences = 30

# Each video will contain 30 frames
sequence_length = 30

# Optional: starting folder index (useful if resuming dataset collection)
start_folder = 0

# Create base directory if it doesn't exist
if not os.path.exists(DATA_PATH):
    os.makedirs(DATA_PATH)

# Create folder structure
for action in actions:
    action_path = os.path.join(DATA_PATH, action)

    if not os.path.exists(action_path):
        os.makedirs(action_path)

    # Create sequence folders (0,1,2,...29)
    for sequence in range(no_sequences):
        seq_path = os.path.join(action_path, str(sequence))

        if not os.path.exists(seq_path):
            os.makedirs(seq_path)

In [ ]:
for action in actions:
    
    action_path = os.path.join(DATA_PATH, action)
    
    # Get existing folders safely
    existing_folders = os.listdir(action_path)
    
    if len(existing_folders) == 0:
        start_index = 0
    else:
        start_index = np.max(np.array(existing_folders).astype(int)) + 1

    for sequence in range(no_sequences):
        try:
            os.makedirs(os.path.join(action_path, str(start_index + sequence)))
        except:
            pass

In [ ]:
cap = cv2.VideoCapture(0)

with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:

    # Loop through actions
    for action in actions:

        # Loop through sequences (videos)
        for sequence in range(no_sequences):

            # Loop through frames in each sequence
            for frame_num in range(sequence_length):

                ret, frame = cap.read()

                # Make detections
                image, results = mediapipe_detection(frame, holistic)

                # Draw landmarks
                draw_styled_landmarks(image, results)

                # Display info
                if frame_num == 0:
                    cv2.putText(image, 'STARTING COLLECTION', (120, 200),
                                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 4, cv2.LINE_AA)

                    cv2.putText(image,
                                f'Collecting {action} Video {sequence}',
                                (15, 12),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.5,
                                (0, 0, 255), 1, cv2.LINE_AA)

                    cv2.imshow('OpenCV Feed', image)
                    cv2.waitKey(500)

                else:
                    cv2.putText(image,
                                f'Collecting {action} Video {sequence}',
                                (15, 12),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.5,
                                (0, 0, 255), 1, cv2.LINE_AA)

                    cv2.imshow('OpenCV Feed', image)

                # Save keypoints
                keypoints = extract_keypoints(results)
                npy_path = os.path.join(DATA_PATH, action, str(sequence), str(frame_num))
                np.save(npy_path, keypoints)

                # Exit condition
                if cv2.waitKey(10) & 0xFF == ord('q'):
                    cap.release()
                    cv2.destroyAllWindows()
                    break

In [ ]:
# your big data collection loop ends here

cap.release()
cv2.destroyAllWindows()

In [ ]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

In [ ]:
label_map = {label:num for num, label in enumerate(actions)}

In [ ]:
label_map

In [ ]:
sequences, labels = [], []

for action in actions:
    action_path = os.path.join(DATA_PATH, action)
    
    # sort and keep only numeric folders
    sequence_list = sorted([s for s in os.listdir(action_path) if s.isdigit()], key=int)

    for sequence in sequence_list:
        window = []
        
        for frame_num in range(sequence_length):
            file_path = os.path.join(action_path, sequence, f"{frame_num}.npy")
            
            if os.path.exists(file_path):
                res = np.load(file_path)
            else:
                res = np.zeros(1662)  # fallback if file missing
            
            window.append(res)

        sequences.append(window)
        labels.append(label_map[action])

In [ ]:
np.array(sequences).shape

In [ ]:
np.array(labels).shape

In [ ]:
X = np.array(sequences)

In [ ]:
X.shape

In [ ]:
y = np.array(labels)

In [ ]:
y = to_categorical(y).astype(int)

In [ ]:
y.shape

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.05,
    random_state=42
)

y_test.shape

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.callbacks import TensorBoard

In [ ]:
log_dir = os.path.join('Logs')
tb_callback = TensorBoard(log_dir=log_dir)

In [ ]:
model = Sequential()

model.add(LSTM(64, return_sequences=True, activation='relu', input_shape=(30, 1662)))
model.add(LSTM(128, return_sequences=True, activation='relu'))
model.add(LSTM(64, return_sequences=False, activation='relu'))

model.add(Dense(64, activation='relu'))
model.add(Dense(32, activation='relu'))

model.add(Dense(actions.shape[0], activation='softmax'))

In [ ]:
model.compile(
    optimizer='Adam',
    loss='categorical_crossentropy',
    metrics=['categorical_accuracy']
)

In [ ]:
model.summary()

In [ ]:
model.fit(X_train, y_train, epochs=2000, callbacks=[tb_callback])

In [ ]:
res = model.predict(X_test)

In [ ]:
actions[np.argmax(res[0])]

In [ ]:
actions[np.argmax(y_test[0])]

In [ ]:
model.save('action.keras')

In [ ]:
from tensorflow.keras.models import load_model

model = load_model('action.keras')

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(ytrue, yhat))

In [ ]:
yhat = model.predict(X_test)

In [ ]:
import numpy as np

ytrue = np.argmax(y_test, axis=1)
yhat = np.argmax(yhat, axis=1)

In [ ]:
from sklearn.metrics import multilabel_confusion_matrix, accuracy_score, classification_report

print(multilabel_confusion_matrix(ytrue, yhat))
print(accuracy_score(ytrue, yhat))
print(classification_report(ytrue, yhat))

In [ ]:
import cv2
import numpy as np

colors = [(245,117,16), (117,245,16), (16,117,245)]

def prob_viz(res, actions, input_frame, colors):
    output_frame = input_frame.copy()

    # Convert res to numpy array (just in case it's list)
    res = np.array(res)

    for num, prob in enumerate(res):
        # Safety check (avoid index error if actions < colors)
        if num >= len(actions):
            break

        # Bar width scaling (better visibility)
        bar_width = int(prob * 300)

        cv2.rectangle(
            output_frame,
            (0, 60 + num * 40),
            (bar_width, 90 + num * 40),
            colors[num % len(colors)],
            -1
        )

        cv2.putText(
            output_frame,
            actions[num],
            (0, 85 + num * 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (255, 255, 255),
            2,
            cv2.LINE_AA
        )

    return output_frame

In [ ]:
res = model.predict(X_test[0:1])[0]


In [ ]:
import matplotlib.pyplot as plt
import cv2

output = prob_viz(res, actions, image, colors)

# Convert BGR → RGB for matplotlib
output = cv2.cvtColor(output, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(18,18))
plt.imshow(output)
plt.axis('off')
plt.show()

In [ ]:
import cv2
import numpy as np

# 1. Variables
sequence = []
sentence = []
predictions = []
threshold = 0.5

cap = cv2.VideoCapture(0)

with mp_holistic.Holistic(min_detection_confidence=0.5,
                           min_tracking_confidence=0.5) as holistic:

    while cap.isOpened():

        # Read feed
        ret, frame = cap.read()

        # Make detections
        image, results = mediapipe_detection(frame, holistic)

        # Draw landmarks
        draw_styled_landmarks(image, results)

        # 2. Prediction logic
        keypoints = extract_keypoints(results)
        sequence.append(keypoints)
        sequence = sequence[-30:]

        if len(sequence) == 30:
            res = model.predict(np.expand_dims(sequence, axis=0))[0]
            pred = np.argmax(res)
            predictions.append(pred)

            # 3. Smoother prediction check
            if np.unique(predictions[-10:])[0] == pred:
                if res[pred] > threshold:

                    word = actions[pred]

                    if len(sentence) == 0 or word != sentence[-1]:
                        sentence.append(word)

            if len(sentence) > 5:
                sentence = sentence[-5:]

            # Probability visualization
            image = prob_viz(res, actions, image, colors)

        # Display sentence
        cv2.rectangle(image, (0, 0), (640, 40), (245, 117, 16), -1)
        cv2.putText(image, ' '.join(sentence),
                    (3, 30),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1,
                    (255, 255, 255),
                    2,
                    cv2.LINE_AA)

        # Show screen
        cv2.imshow('OpenCV Feed', image)

        # Break
        if cv2.waitKey(10) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

In [ ]:
X_test[0],shape

(num_sequences,30,1662)